# Qwen3.5-9B QLoRA Grading with Assistant-Only Loss

This notebook fine-tunes `Qwen/Qwen3.5-9B` using **QLoRA** for rubric-based grading. Training loss is restricted to the assistant completion, so the model learns the JSON score array without being trained to reproduce the question and rubric prompt.

Expected CSV columns:

`id, question, desired_answer, student_answer, criteria[], criteria_score[], criteria_max_score[], student_score`

The model input is:
- question
- desired_answer
- student_answer
- criteria[]

The model target is:
`criteria_score[]`

Example target:
`[1, 0, 1, 1]`

`criteria_max_score[]` and `student_score` are not used as model inputs to avoid leaking grading information.

**NOTE**: Thinking mode is disabled since it eats up all the available `max_new_tokens`, causing an invalid response (not array of criteria_scores). `max_new_tokens` limits time and GPU memory used for the grading task. Experiment with thinking mode if there is time.

## 1. Install dependencies

Installs the Hugging Face training stack, PEFT, bitsandbytes, TRL, pandas, and scikit-learn.

In [1]:
# Install the compiler Triton needs, followed by the Python training stack.
%conda install -y -c conda-forge gcc_linux-64
%pip install -U torch torchvision transformers accelerate peft datasets bitsandbytes trl sentencepiece pandas scikit-learn

2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and GPU check

Imports all libraries and confirms whether CUDA is available. QLoRA training for a 9B model should be done on a supported GPU.

In [2]:
# Import data, model, training, evaluation, and system utilities.
import os
import shutil
import gc
import ast
import csv
import json
import re

# Triton JIT-compiles a small CUDA helper and therefore needs a C compiler.
# Conda's compiler has a target-prefixed name, so expose it through CC.
cc = (
    shutil.which("x86_64-conda-linux-gnu-cc")
    or shutil.which("x86_64-conda_cos7-linux-gnu-cc")
    or shutil.which("gcc")
    or shutil.which("clang")
)
if cc is None:
    raise RuntimeError(
        "No C compiler found. Run the dependency cell above, then restart "
        "the kernel and run the notebook from the beginning."
    )
os.environ["CC"] = cc
print("Triton C compiler:", cc)

import torch
import pandas as pd

from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    )

from trl import SFTConfig, SFTTrainer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

Triton C compiler: /home/ubuntu/anaconda3/envs/lora/bin/x86_64-conda-linux-gnu-cc
PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 22.0 GB


## 3. Configuration

Defines the model, dataset path, output path, maximum sequence length, and LoRA/training hyperparameters.

The base model is loaded only once later in the notebook.

In [3]:
# Centralize paths and hyperparameters so experiments are easy to reproduce.
MODEL_ID = "Qwen/Qwen3.5-9B"

# Change this to your CSV file.
DATASET_PATH = "./training_data.csv"

OUTPUT_DIR = "./qwen35-9b-grading-lora"

# All current examples are at most 424 tokens after chat formatting.
# A 512-token cap preserves every row while limiting worst-case memory use.
MAX_SEQ_LENGTH = 512

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LEARNING_RATE = 1e-4
NUM_EPOCHS = 5
# Process four examples concurrently while accumulating two microbatches.
# 4 physical examples x 2 accumulation steps = effective batch size 8.
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2

## 4. Configure 4-bit quantisation

`bnb_config` controls how the large Qwen base-model weights are loaded into memory.

- `load_in_4bit=True` loads the base weights in 4-bit.
- `nf4` is a common QLoRA quantisation format.
- double quantisation further reduces memory usage.
- computation still happens in BF16 or FP16.

This configuration is used in the model-loading call through `quantization_config=bnb_config`.

In [4]:
# Prefer BF16 on supported GPUs; otherwise use FP16 computation.
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("Compute dtype:", compute_dtype)

Compute dtype: torch.bfloat16


## 5. Load the processor and model

This is the only base-model loading section.

`AutoProcessor` loads tokenisation/chat-template support.

`AutoModelForMultimodalLM` loads the actual model weights.

`quantization_config=bnb_config` is where 4-bit loading is applied.

In [5]:
# Load one processor and one 4-bit base model to minimize GPU memory use.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
    # Use PyTorch's optimized scaled-dot-product attention kernels.
    attn_implementation="sdpa",
    trust_remote_code=True,
)

print("Loaded model:", MODEL_ID)

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Loaded model: Qwen/Qwen3.5-9B


In [6]:
# Summarize parameter storage by dtype and layer class for a quantization check.
from collections import defaultdict

stats = defaultdict(int)

for name, param in model.named_parameters():
    key = (str(param.dtype), param.__class__.__name__)
    stats[key] += param.numel()

for (dtype, cls), count in sorted(stats.items(), key=lambda x: -x[1]):
    print(f"{dtype:20} {cls:20} {count/1e6:10.2f} M params")

print("Loaded in 4bit:", getattr(model, "is_loaded_in_4bit", False))

torch.uint8          Params4bit              3684.84 M params
torch.bfloat16       Parameter               2040.13 M params
Loaded in 4bit: True


## 6. Prepare the quantised model for QLoRA

Prepares the 4-bit model for k-bit training and enables gradient checkpointing to reduce VRAM usage.

The base model remains frozen.

In [7]:
# Freeze the entire base model
for param in model.parameters():
    param.requires_grad = False

# Disable KV cache during training
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print("Base model prepared without BF16 -> FP32 casting")

Base model prepared without BF16 -> FP32 casting


## 7. Detect LoRA target modules

Finds standard attention and MLP projection layers that are commonly adapted by LoRA.

In [8]:
# Detect projection names present in this model rather than assuming its layout.
candidate_suffixes = {
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
}

found = sorted({
    name.split(".")[-1]
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
    and name.split(".")[-1] in candidate_suffixes
})

print("Detected LoRA target modules:", found)

if not found:
    raise RuntimeError(
        "No expected LoRA target modules were detected. "
        "Inspect model.named_modules() and update target_modules."
    )

Detected LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']


## 8. Attach LoRA adapters

`lora_config` controls the small trainable low-rank adapter layers.

Difference:

- `bnb_config` controls how the frozen base model is stored.
- `lora_config` controls what extra parameters are trained.

In [9]:
# Attach trainable low-rank adapters while leaving base weights frozen.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=found,
)

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable params: 29,097,984 || all params: 9,438,911,728 || trainable%: 0.3083


## 9. Load your CSV dataset

Loads the rubric-grading data and checks that the required columns exist.

In [10]:
# Read the CSV and fail early if any model-required fields are absent.
df = pd.read_csv(DATASET_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

required_columns = [
    "question",
    "desired_answer",
    "student_answer",
    "criteria[]",
    "criteria_score[]",
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df.head()

Rows: 2165
Columns: ['id', 'question', 'desired_answer', 'student_answer', 'criteria[]', 'criteria_score[]', 'criteria_max_score[]', 'student_score']


,id,question,desired_answer,student_answer,criteria[],criteria_score[],criteria_max_score[],student_score
0,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,High risk problems are address in the prototyp...,"""[Answer mentions simulating behaviour of port...","""[0]""","""[1]""",0
1,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,To simulate portions of the desired final prod...,"""[Answer mentions simulating behaviour of port...","""[1]""","""[1]""",1
2,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,A prototype program simulates the behaviors of...,"""[Answer mentions simulating behaviour of port...","""[1]""","""[1]""",1
3,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,Defined in the Specification phase a prototype...,"""[Answer mentions simulating behaviour of port...","""[1]""","""[1]""",1
4,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,It is used to let the users have a first idea ...,"""[Answer mentions simulating behaviour of port...","""[0]""","""[1]""",0


## 10. Parse list-valued CSV columns

CSV files usually store arrays as strings. This converts values such as:

`["criterion one", "criterion two"]`

and:

`[1, 0]`

back into Python lists.

In [11]:
# Normalize legacy JSON lists and the dataset's newer quoted bracket format.
def parse_array(value):
    """Parse JSON/Python lists and the dataset's quoted [a,b] format."""
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if not isinstance(value, str):
        raise TypeError(f"Expected list-like string, got {type(value)}")

    text = value.strip()

    # The new CSV encoding unwraps to a quoted string such as "[0,1]".
    # Decode up to two JSON string layers before parsing the actual list.
    for _ in range(2):
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            break
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            text = parsed.strip()
            continue
        break

    # Retain compatibility with old Python-style lists containing quotes.
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            text = parsed.strip()
    except (ValueError, SyntaxError):
        pass

    # New criteria values look like [criterion one,criterion two], with
    # unquoted text entries. Parse the contents as a CSV record.
    if not (text.startswith("[") and text.endswith("]")):
        raise ValueError(f"Expected a bracketed list, got: {value!r}")

    inner = text[1:-1].strip()
    if not inner:
        return []

    # Criteria in this dataset begin with 'Answer'. Split only before that
    # marker so commas inside a criterion (for example 'left, root, right')
    # remain part of the criterion. Numeric arrays use ordinary CSV parsing.
    if re.search(r",(?=Answer\s)", inner):
        items = re.split(r",(?=Answer\s)", inner)
    else:
        items = next(csv.reader([inner], skipinitialspace=True))
    return [item.strip() for item in items]


def parse_criteria(value, expected_count):
    """Parse criteria, using the label count to resolve internal commas."""
    items = parse_array(value)
    if len(items) == expected_count:
        return items

    text = value.strip()
    for _ in range(2):
        try:
            decoded = json.loads(text)
        except json.JSONDecodeError:
            break
        if isinstance(decoded, str):
            text = decoded.strip()
        else:
            break

    if text.startswith("[") and text.endswith("]"):
        inner = text[1:-1].strip()
        items = [inner] if expected_count == 1 else re.split(r",(?=Answer\s)", inner)

    items = [item.strip() for item in items]
    if len(items) != expected_count:
        raise ValueError(
            f"Parsed {len(items)} criteria but expected {expected_count}: {value!r}"
        )
    return items


df["criteria_score"] = df["criteria_score[]"].apply(
    lambda value: [int(x) for x in parse_array(value)]
)
df["criteria"] = df.apply(
    lambda row: parse_criteria(row["criteria[]"], len(row["criteria_score"])),
    axis=1,
)
if "criteria_max_score[]" in df.columns:
    df["criteria_max_score"] = df["criteria_max_score[]"].apply(
        lambda value: [int(x) for x in parse_array(value)]
    )

df[["criteria", "criteria_score"]].head()

,criteria,criteria_score
0,[Answer mentions simulating behaviour of porti...,[0]
1,[Answer mentions simulating behaviour of porti...,[1]
2,[Answer mentions simulating behaviour of porti...,[1]
3,[Answer mentions simulating behaviour of porti...,[1]
4,[Answer mentions simulating behaviour of porti...,[0]


## 11. Validate labels

Each row must have exactly one binary score per criterion.

This checks:
- number of criteria equals number of labels
- every label is 0 or 1

In [12]:
# Validate label alignment before an invalid row can reach the trainer.
def validate_row(row):
    criteria = row["criteria"]
    scores = row["criteria_score"]
    max_scores = row.get("criteria_max_score")

    if not criteria or any(not str(item).strip() for item in criteria):
        return False, "empty criterion found"

    if len(criteria) != len(scores):
        return False, (
            f"criteria length {len(criteria)} "
            f"!= score length {len(scores)}"
        )

    if max_scores is not None and len(criteria) != len(max_scores):
        return False, (
            f"criteria length {len(criteria)} "
            f"!= max-score length {len(max_scores)}"
        )

    invalid_scores = [
        x for x in scores
        if int(x) not in (0, 1)
    ]

    if invalid_scores:
        return False, f"non-binary scores found: {invalid_scores}"

    return True, None


errors = []

for idx, row in df.iterrows():
    valid, reason = validate_row(row)

    if not valid:
        errors.append({
            "row_index": idx,
            "reason": reason,
        })

if errors:
    print("Dataset validation errors:")

    for error in errors[:20]:
        print(error)

    raise ValueError(
        f"Found {len(errors)} invalid dataset rows."
    )

print("All rows passed label validation.")

All rows passed label validation.


## 12. Define the system prompt

The model is trained as a strict rubric grader.

Its output must contain only a JSON array of binary values, for example:

`[1, 0, 1]`

In [13]:
# Constrain supervised targets to one ordered binary value per criterion.
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)

print(SYSTEM_PROMPT)

You are a strict grading assistant.

Your task is to evaluate whether a student's answer fulfills each grading criterion.

For each criterion:
- Output 1 if the criterion is fulfilled.
- Output 0 if the criterion is not fulfilled.

Evaluate each criterion independently.

Return ONLY a valid JSON array containing 0 or 1.
The output array must contain exactly one value for each criterion, in the same order as the criteria provided.

Do not include explanations.
Do not include markdown.
Do not include any text before or after the JSON array.


## 13. Convert each row into a supervised example

The user prompt contains:
1. question
2. desired answer
3. student answer
4. ordered criteria

The assistant target is exactly `criteria_score[]`.

`student_score` and `criteria_max_score[]` are retained only as evaluation metadata. They are excluded from both the model prompt and the training dataset passed to TRL.

In [14]:
# Convert one dataframe row into system, user, and assistant training turns.
def make_training_example(row):
    criteria_text = "\n".join(
        f"{i + 1}. {criterion}"
        for i, criterion in enumerate(row["criteria"])
    )

    user_prompt = (
        f"Question:\n{row['question']}\n\n"
        f"Desired answer:\n{row['desired_answer']}\n\n"
        f"Student answer:\n{row['student_answer']}\n\n"
        f"Criteria:\n{criteria_text}"
    )

    target = json.dumps(
        [int(x) for x in row["criteria_score"]]
    )

    # Keep the conversation split into prompt and completion. TRL can then
    # mask every prompt token and calculate loss only on the assistant turn.
    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]
    completion_messages = [
        {"role": "assistant", "content": [{"type": "text", "text": target}]},
    ]

    return {
        "prompt": prompt_messages,
        "completion": completion_messages,
        # Keep Qwen's template identical when TRL tokenizes the prompt
        # separately and together with the assistant completion.
        "chat_template_kwargs": {"enable_thinking": False},
        # Plain fields are retained only for readable evaluation later.
        "user": user_prompt,
        "assistant": target,
        # Evaluation-only metadata; these values never enter the prompt.
        "criteria_max_score": [int(x) for x in row["criteria_max_score"]],
        "student_score": float(row["student_score"]),
    }


examples = [
    make_training_example(row)
    for _, row in df.iterrows()
]

dataset = Dataset.from_list(examples)

print(dataset[0])

{'prompt': [{'role': 'system', 'content': [{'type': 'text', 'text': "You are a strict grading assistant.\n\nYour task is to evaluate whether a student's answer fulfills each grading criterion.\n\nFor each criterion:\n- Output 1 if the criterion is fulfilled.\n- Output 0 if the criterion is not fulfilled.\n\nEvaluate each criterion independently.\n\nReturn ONLY a valid JSON array containing 0 or 1.\nThe output array must contain exactly one value for each criterion, in the same order as the criteria provided.\n\nDo not include explanations.\nDo not include markdown.\nDo not include any text before or after the JSON array."}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Question:\nWhat is the role of a prototype program in problem solving?\n\nDesired answer:\nTo simulate the behaviour of portions of the desired software product.\n\nStudent answer:\nHigh risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used 

## 14. Split into train, validation, and test sets

Uses an 80/10/10 split with a fixed seed.

The test set is kept completely separate from training.

In [15]:
# Reserve 80% for training and 10% each for validation and final testing.
split_1 = dataset.train_test_split(
    test_size=0.20,
    seed=42,
)

train_dataset = split_1["train"]
temp_dataset = split_1["test"]

split_2 = temp_dataset.train_test_split(
    test_size=0.50,
    seed=42,
)

validation_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Train: 1732
Validation: 216
Test: 217


## 15. Preserve the prompt/completion boundary

Keeps the system and user messages in `prompt` and the JSON assistant turn in `completion`. TRL applies Qwen's chat template later while retaining this boundary, allowing all prompt labels to be masked from loss.

In [16]:
# Preserve prompt and completion as separate conversational columns.
# Evaluation-only metadata remains in test_dataset but is excluded here.
training_columns = ["prompt", "completion", "chat_template_kwargs"]
formatted_train = train_dataset.select_columns(training_columns)
formatted_validation = validation_dataset.select_columns(training_columns)

# Display both portions to make the loss boundary explicit.
print("PROMPT (masked from loss):")
print(processor.apply_chat_template(
    formatted_train[0]["prompt"], tokenize=False, add_generation_prompt=False,
    enable_thinking=False,
))
print("COMPLETION (included in loss):")
print(processor.apply_chat_template(
    formatted_train[0]["prompt"] + formatted_train[0]["completion"],
    tokenize=False, add_generation_prompt=False, enable_thinking=False,
))

PROMPT (masked from loss):
<|im_start|>system
You are a strict grading assistant.

Your task is to evaluate whether a student's answer fulfills each grading criterion.

For each criterion:
- Output 1 if the criterion is fulfilled.
- Output 0 if the criterion is not fulfilled.

Evaluate each criterion independently.

Return ONLY a valid JSON array containing 0 or 1.
The output array must contain exactly one value for each criterion, in the same order as the criteria provided.

Do not include explanations.
Do not include markdown.
Do not include any text before or after the JSON array.<|im_end|>
<|im_start|>user
Question:
What is the advantage of linked lists over arrays?

Desired answer:
Linked lists are dynamic structures, which allow for a variable number of elements to be stored.

Student answer:
A link list can grow to a infinite size as a array has a fixes size.

Criteria:
1. Answer mentions dynamic structure/variable length<|im_end|>

COMPLETION (included in loss):
<|im_start|>sys

## 16. Extract the tokenizer

Qwen3.5 uses a processor, but this is a text-only SFT task. The underlying tokenizer is passed to TRL.

In [17]:
# TRL needs the tokenizer component for truncation, packing, and batching.
tokenizer = getattr(
    processor,
    "tokenizer",
    processor,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer:", tokenizer.__class__.__name__)
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

Tokenizer: Qwen2Tokenizer
EOS token: <|im_end|>
PAD token: <|endoftext|>


## 17. Configure training

This section uses `SFTConfig`, combining Hugging Face training controls with TRL sequence-processing options. The safe optimized configuration uses:

- physical batch size 4 and 2 accumulation steps (effective batch size 8)
- a 512-token cap, covering every currently measured example
- packing and padding-free mode disabled because SDPA cannot safely isolate flattened samples
- paged 8-bit AdamW and mixed precision
- gradient checkpointing enabled to reduce activation-memory usage
- completion-only loss so optimization focuses on the assistant JSON target
- validation and checkpoint saving after every epoch

In [18]:
# SFTConfig extends Hugging Face training arguments with TRL-specific
# sequence preparation options such as max_length and packing.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Training duration and physical/effective batch configuration.
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Optimizer schedule for the trainable LoRA parameters.
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    # In Transformers 5, a fractional warmup_steps value is a ratio.
    warmup_steps=0.05,
    lr_scheduler_type="cosine",

    optim="paged_adamw_8bit",

    # Use the mixed-precision type supported by the current GPU.
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),

    # Recompute selected activations during backward to reduce peak VRAM.
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Calculate loss only on the completion (the assistant JSON array).
    # Qwen3.5's template lacks a generation mask, so prompt/completion
    # masking is used instead of assistant_only_loss=True.
    completion_only_loss=True,

    # SDPA cannot reliably isolate samples flattened by TRL packing. Keep
    # both packing and padding-free mode disabled unless Flash Attention
    # is installed and explicitly selected.
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    padding_free=False,

    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none",
)

## 18. Create the SFT trainer

Builds the TRL trainer from the model, optimized configuration, formatted datasets, and tokenizer. Sequence length and packing are configured in `SFTConfig`, as required by the installed TRL version.

In [19]:
# SFTTrainer tokenizes, packs, batches, evaluates, and checkpoints the run.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    eval_dataset=formatted_validation,
    processing_class=tokenizer,
)

print(trainer)

Tokenizing train dataset:   0%|          | 0/1732 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1732 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1732 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1732 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/216 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/216 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/216 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/216 [00:00<?, ? examples/s]

### Verify the assistant-only loss mask

Before training, inspect one processed sequence. Prompt positions must have label `-100`, which PyTorch ignores during cross-entropy loss. The remaining labels should decode to assistant completions such as `[0]` or `[1, 0]`.

This notebook uses `completion_only_loss=True` because the installed Qwen3.5 chat template does not expose the `{% generation %}` mask required by TRL's `assistant_only_loss=True`. For this one-assistant-turn dataset, completion-only masking provides the intended assistant-only objective.

In [20]:
# Inspect the labels after TRL has applied the template, tokenization,
# completion mask, truncation, and packing.
mask_sample = trainer.train_dataset[0]
mask_labels = mask_sample["labels"]
mask_input_ids = mask_sample["input_ids"]

ignored_tokens = sum(label == -100 for label in mask_labels)
trained_token_ids = [
    token_id for token_id, label in zip(mask_input_ids, mask_labels)
    if label != -100
]

if ignored_tokens == 0:
    raise RuntimeError("Loss-mask check failed: no prompt tokens were masked.")
if not trained_token_ids:
    raise RuntimeError("Loss-mask check failed: the completion is fully masked.")

print("Total tokens in sample:", len(mask_labels))
print("Prompt tokens ignored by loss:", ignored_tokens)
print("Completion tokens trained:", len(trained_token_ids))
print("Decoded trained tokens:")
print(tokenizer.decode(trained_token_ids, skip_special_tokens=False))

Total tokens in sample: 213
Prompt tokens ignored by loss: 208
Completion tokens trained: 5
Decoded trained tokens:
[1]<|im_end|>



## 19. Train the LoRA adapter

Starts supervised fine-tuning.

Only the LoRA parameters are updated; the 4-bit base model remains frozen.

In [21]:
# Execute supervised fine-tuning and retain its runtime/loss summary.
train_result = trainer.train()
train_result

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.085841,0.057147,0.067772,395761.000000,0.977080
2,0.053675,0.044432,0.049462,791522.000000,0.984198
3,0.012433,0.061904,0.016470,1187283.000000,0.982167
4,0.003698,0.071044,0.011675,1583044.000000,0.985081
5,0.000905,0.071200,0.011482,1978805.000000,0.985081


TrainOutput(global_step=1085, training_loss=0.03382308068749611, metrics={'train_runtime': 2197.7236, 'train_samples_per_second': 3.94, 'train_steps_per_second': 0.494, 'total_flos': 1.13704847446176e+17, 'train_loss': 0.03382308068749611, 'epoch': 5.0})

## 20. Save the trained adapter

Saves the LoRA adapter and processor files.

This does not save another full 9B model.

In [22]:
# Save only the compact adapter and processor needed to reproduce inference.
trainer.model.save_pretrained(
    OUTPUT_DIR
)

processor.save_pretrained(
    OUTPUT_DIR
)

print("Saved LoRA adapter to:", OUTPUT_DIR)

Saved LoRA adapter to: ./qwen35-9b-grading-lora


## 21. Reuse the same model for inference

No second model-loading call is needed.

The already-loaded and fine-tuned model is reused directly.

In [23]:
# Release temporary training allocations and prepare deterministic generation.
gc.collect()
torch.cuda.empty_cache()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = True

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1152)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-26): 27 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
              (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear4bit(in_features=1152, out_features=3456, bias=True)
                (proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear4bit(in_features=1152, out_feature

## 22. Prediction function

Generates a binary criteria array for a new example.

The result is validated to ensure:
- valid JSON
- list output
- correct number of labels
- labels are only 0 or 1

In [24]:
# Build a grading prompt, generate one score array, and validate its structure.
def predict_criteria_scores(
    question,
    desired_answer,
    student_answer,
    criteria,
):
    criteria_text = "\n".join(
        f"{i + 1}. {criterion}"
        for i, criterion in enumerate(criteria)
    )

    user_prompt = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_prompt,
                }
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )

    input_device = next(model.parameters()).device

    inputs = {
        key: value.to(input_device)
        if hasattr(value, "to")
        else value
        for key, value in inputs.items()
    }

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max(128, len(criteria) * 8),
            do_sample=False,
        )

    prompt_length = inputs["input_ids"].shape[-1]

    answer_ids = generated[0][prompt_length:]

    raw_output = processor.decode(
        answer_ids,
        skip_special_tokens=True,
    ).strip()

    # Prefer the exact response, but tolerate an unwanted reasoning preamble.
    # Restrict fallback matches to binary arrays so arbitrary prose is ignored.
    try:
        scores = json.loads(raw_output)
    except json.JSONDecodeError:
        candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw_output)
        matching = []
        for candidate in candidates:
            parsed = json.loads(candidate)
            if len(parsed) == len(criteria):
                matching.append(parsed)
        scores = matching[-1] if matching else None

    if scores is None:
        return {
            "valid": False,
            "raw_output": raw_output,
            "scores": None,
            "error": "Output was not valid JSON.",
        }

    if not isinstance(scores, list):
        return {
            "valid": False,
            "raw_output": raw_output,
            "scores": None,
            "error": "Output JSON was not a list.",
        }

    try:
        scores = [int(x) for x in scores]
    except (TypeError, ValueError):
        return {
            "valid": False,
            "raw_output": raw_output,
            "scores": None,
            "error": "Output contained non-integer values.",
        }

    if len(scores) != len(criteria):
        return {
            "valid": False,
            "raw_output": raw_output,
            "scores": scores,
            "error": (
                f"Expected {len(criteria)} scores "
                f"but received {len(scores)}."
            ),
        }

    if any(x not in (0, 1) for x in scores):
        return {
            "valid": False,
            "raw_output": raw_output,
            "scores": scores,
            "error": "Output contained values other than 0 or 1.",
        }

    return {
        "valid": True,
        "raw_output": raw_output,
        "scores": scores,
        "error": None,
    }

## 23. Manual prediction example

Replace this with one of your own unseen grading examples.

In [25]:
# Smoke-test inference with a small example whose expected result is [1, 0, 0].
result = predict_criteria_scores(
    question="Explain why HTTPS is more secure than HTTP.",
    desired_answer="HTTPS encrypts network traffic using TLS.",
    student_answer="HTTPS protects transmitted data using encryption.",
    criteria=[
        "Mentions encryption",
        "Mentions TLS",
        "Claims HTTPS sends plaintext",
    ],
)

print(result)

{'valid': True, 'raw_output': '[1, 0, 0]', 'scores': [1, 0, 0], 'error': None}


## 24. Evaluate on the held-out test set

This first evaluation measures the trained LoRA adapter. Section 26 repeats the same evaluation with the adapter disabled to produce a baseline comparison.

Reports:
- exact-match accuracy
- per-criterion accuracy
- precision
- recall
- F1
- invalid output rate

In [26]:
# Evaluate the fine-tuned adapter on examples never used for optimization.
all_true = []
all_pred = []

exact_matches = []
invalid_count = 0

evaluation_rows = []


for example in test_dataset:
    user_text = example["user"]

    def extract_section(text, start_marker, end_marker=None):
        start = text.index(start_marker) + len(start_marker)

        if end_marker is None:
            return text[start:].strip()

        end = text.index(end_marker, start)
        return text[start:end].strip()

    question = extract_section(
        user_text,
        "Question:\n",
        "\n\nDesired answer:"
    )

    desired_answer = extract_section(
        user_text,
        "Desired answer:\n",
        "\n\nStudent answer:"
    )

    student_answer = extract_section(
        user_text,
        "Student answer:\n",
        "\n\nCriteria:"
    )

    criteria_block = extract_section(
        user_text,
        "Criteria:\n"
    )

    criteria = []

    for line in criteria_block.splitlines():
        line = line.strip()

        if not line:
            continue

        if ". " in line:
            criteria.append(
                line.split(". ", 1)[1]
            )
        else:
            criteria.append(line)

    true_scores = [
        int(x)
        for x in json.loads(example["assistant"])
    ]

    prediction = predict_criteria_scores(
        question=question,
        desired_answer=desired_answer,
        student_answer=student_answer,
        criteria=criteria,
    )

    if not prediction["valid"]:
        invalid_count += 1

        evaluation_rows.append({
            "ground_truth": true_scores,
            "prediction": prediction["raw_output"],
            "valid": False,
        })

        continue

    pred_scores = prediction["scores"]

    exact_matches.append(
        pred_scores == true_scores
    )

    all_true.extend(true_scores)
    all_pred.extend(pred_scores)

    evaluation_rows.append({
        "ground_truth": true_scores,
        "prediction": pred_scores,
        "valid": True,
    })


if len(test_dataset) > 0:
    invalid_output_rate = (
        invalid_count / len(test_dataset)
    )
else:
    invalid_output_rate = 0.0

print("Invalid output rate:", invalid_output_rate)

if all_true:
    exact_match_accuracy = (
        sum(exact_matches) / len(exact_matches)
        if exact_matches
        else 0.0
    )

    print("Exact-match accuracy:", exact_match_accuracy)

    print(
        "Per-criterion accuracy:",
        accuracy_score(all_true, all_pred)
    )

    print(
        "Precision:",
        precision_score(
            all_true,
            all_pred,
            zero_division=0,
        )
    )

    print(
        "Recall:",
        recall_score(
            all_true,
            all_pred,
            zero_division=0,
        )
    )

    print(
        "F1:",
        f1_score(
            all_true,
            all_pred,
            zero_division=0,
        )
    )

Invalid output rate: 0.0
Exact-match accuracy: 0.8617511520737328
Per-criterion accuracy: 0.9098143236074271
Precision: 0.9519650655021834
Recall: 0.9045643153526971
F1: 0.9276595744680851


## 25. Inspect individual predictions

Creates a dataframe showing the ground-truth array and model prediction for each evaluated test example.

In [27]:
# Convert the detailed prediction records into a table for error analysis.
evaluation_df = pd.DataFrame(
    evaluation_rows
)

evaluation_df.head(20)

,ground_truth,prediction,valid
0,"[1, 1]","[1, 1]",True
1,"[0, 0]","[0, 0]",True
2,[1],[1],True
3,"[0, 0]","[0, 0]",True
4,[1],[1],True
5,[1],[1],True
6,"[0, 0, 1]","[0, 0, 1]",True
7,[0],[0],True
8,"[1, 0, 1]","[1, 0, 0]",True
9,"[1, 1]","[1, 1]",True


## 26. Compare the fine-tuned adapter with the baseline model

A fair baseline measures the original frozen Qwen model on the **same held-out test set** and with the same prompt, decoding settings, parser, and metrics. The only difference is whether the trained LoRA adapter is active.

`model.disable_adapter()` temporarily bypasses LoRA without loading a second 9B model, so the comparison is safe for GPU memory. Once the context exits, the fine-tuned adapter is active again. Exact-match accuracy counts invalid responses as incorrect, while per-criterion metrics use valid predictions whose lengths match the labels.

Final scores are reconstructed as `sum(predicted criterion × criterion maximum)`. The comparison reports:

- **final-score exact accuracy:** fraction of responses with the exact numeric score
- **within-one accuracy:** fraction no more than one point away
- **MAE:** average absolute point difference; lower is better
- **RMSE:** like MAE but penalizes larger mistakes more strongly; lower is better

Invalid generations count as incorrect for exact and within-one accuracy. MAE and RMSE are calculated only for valid numeric predictions and must therefore be interpreted alongside invalid-output rate. Fine-tuned predictions are reused from Section 24, so this section performs only one additional inference pass for the baseline.

In [28]:
def unpack_test_example(example):
    """Recover the four prompt fields and labels from one test example."""
    user_text = example["user"]

    def section(start_marker, end_marker=None):
        start = user_text.index(start_marker) + len(start_marker)
        end = user_text.index(end_marker, start) if end_marker else len(user_text)
        return user_text[start:end].strip()

    question = section("Question:\n", "\n\nDesired answer:")
    desired_answer = section("Desired answer:\n", "\n\nStudent answer:")
    student_answer = section("Student answer:\n", "\n\nCriteria:")
    criteria_block = section("Criteria:\n")

    # Remove the numeric prefixes added when the prompt was constructed.
    criteria = [
        line.strip().split(". ", 1)[-1]
        for line in criteria_block.splitlines()
        if line.strip()
    ]
    true_scores = [int(x) for x in json.loads(example["assistant"])]
    max_scores = [int(x) for x in example["criteria_max_score"]]
    true_final_score = float(example["student_score"])
    return (
        question, desired_answer, student_answer, criteria, true_scores,
        max_scores, true_final_score,
    )


def evaluate_current_model(dataset, model_name, cached_rows=None):
    """Generate predictions and calculate grading metrics for one model state."""
    criterion_true = []
    criterion_pred = []
    exact_correct = 0
    invalid_count = 0
    final_score_errors = []
    final_score_exact = 0
    final_score_within_one = 0
    rows = []

    for index, example in enumerate(dataset):
        (
            question, desired, student, criteria, true_scores,
            max_scores, true_final_score,
        ) = unpack_test_example(example)
        if cached_rows is None:
            prediction = predict_criteria_scores(question, desired, student, criteria)
        else:
            # Reuse the fine-tuned predictions produced in Section 24.
            cached = cached_rows[index]
            prediction = {
                "valid": cached["valid"],
                "scores": cached["prediction"] if cached["valid"] else None,
                "raw_output": cached["prediction"],
            }

        # Invalid arrays are failures for response-level exact matching.
        is_valid = prediction["valid"]
        pred_scores = prediction["scores"] if is_valid else None
        is_exact = bool(is_valid and pred_scores == true_scores)
        exact_correct += int(is_exact)
        invalid_count += int(not is_valid)

        # Criterion metrics require aligned arrays, so invalid arrays are omitted.
        if is_valid:
            criterion_true.extend(true_scores)
            criterion_pred.extend(pred_scores)

            # Convert binary criterion decisions into the final numeric score.
            predicted_final_score = sum(
                score * maximum for score, maximum in zip(pred_scores, max_scores)
            )
            score_error = abs(predicted_final_score - true_final_score)
            final_score_errors.append(score_error)
            final_score_exact += int(score_error == 0)
            final_score_within_one += int(score_error <= 1)
        else:
            predicted_final_score = None
            score_error = None

        rows.append({
            "model": model_name,
            "ground_truth": true_scores,
            "prediction": pred_scores if is_valid else prediction["raw_output"],
            "valid": is_valid,
            "exact_match": is_exact,
            "true_final_score": true_final_score,
            "predicted_final_score": predicted_final_score,
            "final_score_absolute_error": score_error,
        })

    total = len(dataset)
    metrics = {
        "model": model_name,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        "exact_match_accuracy": exact_correct / total if total else 0.0,
        "per_criterion_accuracy": accuracy_score(criterion_true, criterion_pred) if criterion_true else 0.0,
        "precision": precision_score(criterion_true, criterion_pred, zero_division=0) if criterion_true else 0.0,
        "recall": recall_score(criterion_true, criterion_pred, zero_division=0) if criterion_true else 0.0,
        "f1": f1_score(criterion_true, criterion_pred, zero_division=0) if criterion_true else 0.0,
        # Invalid generations count as incorrect for the two score accuracies.
        "final_score_exact_accuracy": final_score_exact / total if total else 0.0,
        "final_score_within_one_accuracy": final_score_within_one / total if total else 0.0,
        # MAE and RMSE describe closeness among valid numeric predictions.
        "final_score_mae": (
            sum(final_score_errors) / len(final_score_errors)
            if final_score_errors else float("nan")
        ),
        "final_score_rmse": (
            (sum(error ** 2 for error in final_score_errors) / len(final_score_errors)) ** 0.5
            if final_score_errors else float("nan")
        ),
    }
    return metrics, rows


# Reuse the trained-adapter predictions already generated in Section 24.
finetuned_metrics, finetuned_rows = evaluate_current_model(
    test_dataset, "Fine-tuned QLoRA", cached_rows=evaluation_rows
)

# Disable LoRA only for this block to measure the untouched base model.
if not hasattr(model, "disable_adapter"):
    raise RuntimeError("This PEFT model does not support temporary adapter disabling.")

with model.disable_adapter():
    baseline_metrics, baseline_rows = evaluate_current_model(
        test_dataset, "Baseline Qwen3.5-9B"
    )

# Display models as rows and metrics as columns for direct comparison.
comparison_df = pd.DataFrame([baseline_metrics, finetuned_metrics]).set_index("model")
comparison_df.loc["Absolute improvement"] = (
    comparison_df.loc["Fine-tuned QLoRA"]
    - comparison_df.loc["Baseline Qwen3.5-9B"]
)

# Keep response-level final scores available for inspecting individual errors.
final_score_predictions_df = pd.DataFrame(baseline_rows + finetuned_rows)
display(comparison_df)
final_score_predictions_df[[
    "model", "true_final_score", "predicted_final_score",
    "final_score_absolute_error", "valid",
]].head(20)


,invalid_output_rate,exact_match_accuracy,per_criterion_accuracy,precision,recall,f1,final_score_exact_accuracy,final_score_within_one_accuracy,final_score_mae,final_score_rmse
model,,,,,,,,,,
Baseline Qwen3.5-9B,0.0,0.668203,0.785146,0.830579,0.834025,0.832298,0.677419,0.972350,0.354839,0.654654
Fine-tuned QLoRA,0.0,0.861751,0.909814,0.951965,0.904564,0.927660,0.866359,0.986175,0.147465,0.418468
Absolute improvement,0.0,0.193548,0.124668,0.121387,0.070539,0.095361,0.188940,0.013825,-0.207373,-0.236186


,model,true_final_score,predicted_final_score,final_score_absolute_error,valid
0,Baseline Qwen3.5-9B,2.0,2,0.0,True
1,Baseline Qwen3.5-9B,0.0,0,0.0,True
2,Baseline Qwen3.5-9B,1.0,1,0.0,True
3,Baseline Qwen3.5-9B,0.0,0,0.0,True
4,Baseline Qwen3.5-9B,1.0,1,0.0,True
5,Baseline Qwen3.5-9B,1.0,1,0.0,True
6,Baseline Qwen3.5-9B,1.0,2,1.0,True
7,Baseline Qwen3.5-9B,0.0,0,0.0,True
8,Baseline Qwen3.5-9B,2.0,1,1.0,True
9,Baseline Qwen3.5-9B,2.0,2,0.0,True
